# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import os
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Authenticate with Hugging Face Hub securely
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    from huggingface_hub import notebook_login

    notebook_login()

# 2. Load Warehouse Dataset for Decline Recovery Lane
DATASET_ID = "FlyRank/internship-warehouse"
TABLE_NAME = (
    "dim_clients"  # Adapt to your lane's table name if specified in repo docs
)

print(f"Loading table '{TABLE_NAME}' from {DATASET_ID}...")
hf_dataset = load_dataset(DATASET_ID, TABLE_NAME, split="train")
df = hf_dataset.to_pandas()

# 3. Filter for March 2026 Mid-Panel Month & Ready-State Availability
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df_slice = df[
        (df["date"] >= "2026-03-01")
        & (df["date"] <= "2026-03-31")
        & (df["is_available"] == True)
    ].copy()
else:
    df_slice = df.copy()

print(f"Successfully loaded {len(df_slice)} rows for feature analysis.")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Define target and honest feature sets
target = "is_recovered"  # Binary label: 1 = Recovered, 0 = Did not recover
features = [
    "decline_magnitude_pct",
    "days_since_decline",
    "pre_decline_position_avg",
    "query_intent_type",
    "page_content_age_days",
]

# 1. DELIBERATE TRAP: Introduce future leakage (e.g., post-drop recovery window clicks ratio)
# This simulates adding a feature that relies on outcomes from the future (t + 30 days).
df_slice["LEAK_future_recovery_clicks_ratio"] = (
    df_slice["post_30d_clicks"] / df_slice["pre_decline_clicks"]
)
leaky_features = features + ["LEAK_future_recovery_clicks_ratio"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df_slice[leaky_features],
    df_slice[target],
    test_size=0.2,
    random_state=42,
)

# Train Leaky Model
leaky_model = RandomForestClassifier(n_estimators=50, random_state=42)
leaky_model.fit(X_train, y_train)

leaky_preds = leaky_model.predict(X_test)
leaky_probs = leaky_model.predict_proba(X_test)[:, 1]

print("--- 🚨 LEAKY MODEL PERFORMANCE ---")
print(f"ROC-AUC: {roc_auc_score(y_test, leaky_probs):.4f} (Artificially inflated)")
print(f"F1-Score: {f1_score(y_test, leaky_preds):.4f}\n")

# 2. THE FIX: Remove the leaky column and train an honest baseline model
X_train_clean = X_train[features]
X_test_clean = X_test[features]

honest_model = RandomForestClassifier(n_estimators=50, random_state=42)
honest_model.fit(X_train_clean, y_train)

honest_preds = honest_model.predict(X_test_clean)
honest_probs = honest_model.predict_proba(X_test_clean)[:, 1]

print("-- ✅ HONEST BASELINE PERFORMANCE ---")
print(
    f"ROC-AUC: {roc_auc_score(y_test, honest_probs):.4f} (Realistic generalization)"
)
print(f"F1-Score: {f1_score(y_test, honest_preds):.4f}")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.